In [17]:
import numpy as np
import evaluate

from datasets import load_dataset
from transformers import AutoTokenizer,AutoModelForTokenClassification,DataCollatorForTokenClassification,TrainingArguments,Trainer, pipeline


In [18]:
dataset = load_dataset("tomaarsen/conll2003")

dataset

DatasetDict({
    train: Dataset({
        features: ['id', 'document_id', 'sentence_id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 14041
    })
    validation: Dataset({
        features: ['id', 'document_id', 'sentence_id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 3250
    })
    test: Dataset({
        features: ['id', 'document_id', 'sentence_id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 3453
    })
})

In [21]:
label_names = dataset["train"].features["ner_tags"].feature.names

label_names

['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC', 'B-MISC', 'I-MISC']

In [22]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")

In [23]:
def tokenize_and_align_labels(examples):

    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True
    )

    labels = []

    for i, label in enumerate(examples["ner_tags"]):

        word_ids = tokenized_inputs.word_ids(batch_index=i)

        previous_word_idx = None

        label_ids = []

        for word_idx in word_ids:

            if word_idx is None:
                label_ids.append(-100)

            elif word_idx != previous_word_idx:
                label_ids.append(label[word_idx])

            else:
                label_ids.append(label[word_idx])

            previous_word_idx = word_idx

        labels.append(label_ids)

    tokenized_inputs["labels"] = labels

    return tokenized_inputs

In [24]:
tokenized_datasets = dataset.map(
    tokenize_and_align_labels,
    batched=True
)

Map:   0%|          | 0/3250 [00:00<?, ? examples/s]

In [25]:
model = AutoModelForTokenClassification.from_pretrained("bert-base-cased",num_labels=len(label_names))

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: bert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized beca

In [27]:
data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

In [29]:
metric = evaluate.load("seqeval")

In [30]:
def compute_metrics(p):

    predictions, labels = p

    predictions = np.argmax(predictions, axis=2)

    true_predictions = [
        [
            label_names[p]
            for (p, l) in zip(prediction, label)
            if l != -100
        ]
        for prediction, label in zip(predictions, labels)
    ]

    true_labels = [
        [
            label_names[l]
            for (p, l) in zip(prediction, label)
            if l != -100
        ]
        for prediction, label in zip(predictions, labels)
    ]

    results = metric.compute(
        predictions=true_predictions,
        references=true_labels
    )

    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }

In [31]:
training_args = TrainingArguments(
    output_dir="./bert-ner-model",

    eval_strategy="epoch",

    save_strategy="epoch",

    learning_rate=5e-5,

    per_device_train_batch_size=16,

    per_device_eval_batch_size=16,

    num_train_epochs=3,

    weight_decay=0.01,

    logging_steps=100,

    push_to_hub=False
)

In [32]:
trainer = Trainer(
    model=model,

    args=training_args,

    train_dataset=tokenized_datasets["train"],

    eval_dataset=tokenized_datasets["validation"],

    data_collator=data_collator,

    compute_metrics=compute_metrics
)

trainer.train()

Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,0.068452,0.068914,0.915442,0.921766,0.918593,0.980279
2,0.035147,0.063834,0.944454,0.939709,0.942076,0.985062
3,0.014196,0.058743,0.941313,0.945451,0.943378,0.985916


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=2634, training_loss=0.06392525834573791, metrics={'train_runtime': 593.906, 'train_samples_per_second': 70.925, 'train_steps_per_second': 4.435, 'total_flos': 1050534559887048.0, 'train_loss': 0.06392525834573791, 'epoch': 3.0})

In [33]:
results = trainer.evaluate()

results

{'eval_loss': 0.058743417263031006,
 'eval_precision': 0.9413130861991961,
 'eval_recall': 0.9454512829714696,
 'eval_f1': 0.9433776464795668,
 'eval_accuracy': 0.985915700241361,
 'eval_runtime': 11.6558,
 'eval_samples_per_second': 278.832,
 'eval_steps_per_second': 17.502,
 'epoch': 3.0}

In [34]:
trainer.save_model("./final-ner-model")

tokenizer.save_pretrained("./final-ner-model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./final-ner-model/tokenizer_config.json', './final-ner-model/tokenizer.json')

In [35]:
ner_pipeline = pipeline(
    "ner",
    model="./final-ner-model",
    tokenizer=tokenizer,
    aggregation_strategy="simple"
)

sentence = "Sundar Pichai is the CEO of Google in California."

predictions = ner_pipeline(sentence)

predictions

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[{'entity_group': 'LABEL_1',
  'score': np.float32(0.99946094),
  'word': 'Sundar',
  'start': 0,
  'end': 6},
 {'entity_group': 'LABEL_2',
  'score': np.float32(0.99882466),
  'word': 'Pichai',
  'start': 7,
  'end': 13},
 {'entity_group': 'LABEL_0',
  'score': np.float32(0.9999026),
  'word': 'is the CEO of',
  'start': 14,
  'end': 27},
 {'entity_group': 'LABEL_3',
  'score': np.float32(0.99835145),
  'word': 'Google',
  'start': 28,
  'end': 34},
 {'entity_group': 'LABEL_0',
  'score': np.float32(0.9997658),
  'word': 'in',
  'start': 35,
  'end': 37},
 {'entity_group': 'LABEL_5',
  'score': np.float32(0.9993315),
  'word': 'California',
  'start': 38,
  'end': 48},
 {'entity_group': 'LABEL_0',
  'score': np.float32(0.999913),
  'word': '.',
  'start': 48,
  'end': 49}]